# 00 — Data Acquisition & Provenance

**Purpose:**  
1. Verify all user-downloaded files are present with correct schema.  
2. Download TARGET-OS, reference files, and (optionally) GEO cohorts.  
3. Record SHA-256 checksums and provenance for every file.

**Do not re-run** if all expected files already exist with matching checksums (the skip logic below handles this).

**Output:** updated `docs/data_provenance.md`, `data/raw/target_os/`, `data/raw/reference/`

In [ ]:
import sys, random, json, hashlib, os, time
from pathlib import Path

random.seed(42)
NOTEBOOK_NAME = '00_download_data'
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
print(f'Run timestamp: {RUN_TIMESTAMP}')

# Works whether JupyterLab was launched from the project root or from notebooks/
_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import sha256_file, file_mtime_iso, log_versions, verify_file

vers = log_versions()

## Phase 1A — Verify DepMap files

In [2]:
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # adjust if needed
DEPMAP_DIR = ROOT / "data" / "raw" / "depmap"

In [3]:
import pandas as pd

fpath = DEPMAP_DIR / 'CRISPRGeneEffect.csv'

# Read just the first 3 rows and the header
sample = pd.read_csv(fpath, nrows=3)
print('Columns (first 5):', list(sample.columns[:5]))
print('Index column name:', sample.columns[0])
print('First column values:', sample.iloc[:, 0].tolist())
print('Shape sample:', sample.shape)

Columns (first 5): ['Unnamed: 0', 'A1BG (1)', 'A1CF (29974)', 'A2M (2)', 'A2ML1 (144568)']
Index column name: Unnamed: 0
First column values: ['ACH-000029', 'ACH-000030', 'ACH-000074']
Shape sample: (3, 18532)


In [4]:
import pandas as pd

DEPMAP_DIR = ROOT / 'data' / 'raw' / 'depmap'

# Files where the ModelID is in an unnamed first column (DepMap convention for matrix files)
MODELID_INDEXED = {'CRISPRGeneEffect.csv'}

EXPECTED_DEPMAP = {
    'Model.csv': [
        'ModelID', 'CellLineName', 'OncotreePrimaryDisease', 'OncotreeLineage'
    ],
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv': [
        'ModelID'
    ],
    'OmicsSomaticMutations.csv': [
        'ModelID', 'HugoSymbol', 'VariantType'
    ],
    'OmicsSomaticMutationsMatrixDamaging.csv': [
        'ModelID'
    ],
    'OmicsCNGeneWGS.csv': [
        'ModelID'
    ],
    'OmicsCNSegmentsWGS.csv': [
        'ModelID', 'Chrom', 'Start', 'End', 'State'
    ],
    'OmicsFusionFiltered.csv': [
        'ModelID', 'Gene1', 'Gene2', 'CanonicalFusionName'
    ],
    'CRISPRGeneEffect.csv': [],  # ModelID is in unnamed index column; verified separately
}

missing, schema_errors, provenance_records = [], [], []

for fname, required_cols in EXPECTED_DEPMAP.items():
    fpath = DEPMAP_DIR / fname
    if not fpath.exists():
        missing.append(fname)
        print(f'  MISSING: {fname}')
        continue

    checksum = sha256_file(fpath)
    mtime = file_mtime_iso(fpath)

    bad = []

    if fname in MODELID_INDEXED:
        # Verify the unnamed first column contains ACH- ModelIDs
        first_col = pd.read_csv(fpath, usecols=[0], nrows=5)
        first_values = first_col.iloc[:, 0].astype(str).tolist()
        if all(v.startswith('ACH-') for v in first_values):
            print(f'  OK  {fname}  (ModelID-indexed, unnamed header)  '
                  f'sha256={checksum[:12]}…  mtime={mtime}')
        else:
            bad = [f'first column does not contain ACH- IDs (got {first_values[:3]})']
            schema_errors.append(f'{fname}: {bad[0]}')
            print(f'  SCHEMA ERROR in {fname}: {bad[0]}')
    else:
        # Standard column-header check
        header = pd.read_csv(fpath, nrows=0)
        header_cols = set(header.columns)
        bad = [c for c in required_cols if c not in header_cols]
        if bad:
            schema_errors.append(f'{fname}: missing columns {bad}')
            print(f'  SCHEMA ERROR in {fname}: missing {bad}')
        else:
            print(f'  OK  {fname}  sha256={checksum[:12]}…  mtime={mtime}')

    provenance_records.append({
        'file': str(fpath.relative_to(ROOT)),
        'source': 'manually downloaded by user (DepMap 26Q1)',
        'depmap_release': '26Q1',
        'mtime': mtime,
        'sha256': checksum,
        'schema_ok': len(bad) == 0,
        'recorded_at': RUN_TIMESTAMP,
    })

if missing:
    print(f'\nMISSING FILES: {missing}')
if schema_errors:
    print(f'\nSCHEMA ERRORS:')
    for e in schema_errors:
        print(f'  {e}')
if not missing and not schema_errors:
    print('\nAll DepMap files present and schema OK.')

  OK  Model.csv  sha256=ea4e0b2a3bc8…  mtime=2026-04-27T20:47:39Z
  OK  OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv  sha256=0377be80c525…  mtime=2026-04-27T20:57:01Z
  OK  OmicsSomaticMutations.csv  sha256=4d5063437357…  mtime=2026-04-27T20:57:52Z
  OK  OmicsSomaticMutationsMatrixDamaging.csv  sha256=45709b1b842d…  mtime=2026-04-27T20:57:56Z
  OK  OmicsCNGeneWGS.csv  sha256=d2fe65b51680…  mtime=2026-04-27T20:58:15Z
  OK  OmicsCNSegmentsWGS.csv  sha256=690e861b3a35…  mtime=2026-04-27T21:00:08Z
  OK  OmicsFusionFiltered.csv  sha256=5087fd478656…  mtime=2026-04-27T20:59:39Z
  OK  CRISPRGeneEffect.csv  (ModelID-indexed, unnamed header)  sha256=e610a4cefb13…  mtime=2026-04-27T20:58:49Z

All DepMap files present and schema OK.


## Phase 1B — Verify literature tables

In [5]:
LIT_DIR = ROOT / 'data' / 'raw' / 'literature_tables'

EXPECTED_LIT = {
    'de_Nonneville_2022_table_EV3.csv':       'peer_reviewed',
    'tcab1_pdx_labels.csv':                   'preprint',
    'tcab1_pdx_sv_calls.csv':                 'preprint',
    'taglialatela_2026_celllines.csv':        'preprint',
    'ALTitude-table3.csv':                    'peer_reviewed',
    'ALTitude-table2,4,5-sh1.csv':            'peer_reviewed',
    'ALTitude-table2,4,5-sh2.csv':            'peer_reviewed',
    'ALTitude-table2,4,5-sh3.csv':            'peer_reviewed',
    'ALTitude-table1.xlsx':                   'peer_reviewed',
    'ALTitude-table2,4,5.xlsx':               'peer_reviewed',
    'ALTitude-table3.xlsx':                   'peer_reviewed',
    'Top3A_EV3.xlsx':                         'peer_reviewed',
    # Chen et al. 2014, Cell Reports — PCGP osteosarcoma cohort.
    # GRCh37 coordinates are classified IN-FRAME by sv_annotator (no liftover).
    'chen_2014_tableS3_SVs.xlsx':             'peer_reviewed',
    'chen_2014_tableS4_TP53_SVs.xlsx':        'peer_reviewed',
    'chen_2014_tableS6_ATRX_ALT.xlsx':        'peer_reviewed',
}

lit_issues = []
for fname, confidence in EXPECTED_LIT.items():
    fpath = LIT_DIR / fname
    if not fpath.exists():
        lit_issues.append(f'MISSING: {fname}')
        continue
    checksum = sha256_file(fpath)
    mtime = file_mtime_iso(fpath)
    print(f'  OK  {fname}  [{confidence}]  sha256={checksum[:12]}…')
    provenance_records.append({
        'file': str(fpath.relative_to(ROOT)),
        'source': f'manually downloaded by user — {fname}',
        'source_confidence': confidence,
        'mtime': mtime,
        'sha256': checksum,
        'recorded_at': RUN_TIMESTAMP,
    })

if lit_issues:
    print(f'\nLiterature table issues: {lit_issues}')
else:
    print('\nAll literature files present.')

print()
print('ALTitude table content summary (from prior inspection):')
print('  table1.xlsx Sheet1       → 959 cell lines, Curated.Marker = gold-standard ALT calls')
print('  table2,4,5.xlsx sheets   → model performance metrics only (no cell-line data)')
print('  table3.xlsx              → ablation study results (no cell-line data)')
print('  CSV exports (sh1-sh3)    → same as table2,4,5 sheets')

  OK  de_Nonneville_2022_table_EV3.csv  [peer_reviewed]  sha256=eff049421d7d…
  OK  tcab1_pdx_labels.csv  [preprint]  sha256=42fe4b1ca426…
  OK  tcab1_pdx_sv_calls.csv  [preprint]  sha256=c132a0011ee5…
  OK  taglialatela_2026_celllines.csv  [preprint]  sha256=b652a5181de0…
  OK  ALTitude-table3.csv  [peer_reviewed]  sha256=c29b3844d156…
  OK  ALTitude-table2,4,5-sh1.csv  [peer_reviewed]  sha256=be6062fa2250…
  OK  ALTitude-table2,4,5-sh2.csv  [peer_reviewed]  sha256=c0eea5dbe894…
  OK  ALTitude-table2,4,5-sh3.csv  [peer_reviewed]  sha256=d1e402e3bdf1…
  OK  ALTitude-table1.xlsx  [peer_reviewed]  sha256=f56d317d7386…
  OK  ALTitude-table2,4,5.xlsx  [peer_reviewed]  sha256=f15c23eab893…
  OK  ALTitude-table3.xlsx  [peer_reviewed]  sha256=f9f6f52d9cf4…
  OK  Top3A_EV3.xlsx  [peer_reviewed]  sha256=fb68a09d13eb…
  OK  chen_2014_tableS3_SVs.xlsx  [peer_reviewed]  sha256=b2b4bba0dded…
  OK  chen_2014_tableS4_TP53_SVs.xlsx  [peer_reviewed]  sha256=d06157011b5e…
  OK  chen_2014_tableS6_ATRX_AL

## Phase 1C — Verify CCLE 2019 SvABA structural variants

Substitutes for ``OmicsStructuralVariants.csv`` which is absent from the DepMap 26Q1 public release.
Reference genome: GRCh37 / hg19 — lifted to GRCh38 inside the SV annotator at use-time.
Caller: SvABA. Source: Cancer Cell Line Encyclopedia, Ghandi et al. 2019, *Nature*.


In [6]:
import pandas as pd

CCLE_DIR = ROOT / 'data' / 'raw' / 'ccle'
EXPECTED_CCLE = {
    'CCLE_translocations_SvABA_20181221.xlsx': {
        'source': 'CCLE 2019 (Ghandi et al., Nature 2019)',
        'reference_genome': 'GRCh37',
        'caller': 'SvABA',
        'note': 'Substitutes for missing DepMap 26Q1 OmicsStructuralVariants.csv',
        'source_confidence': 'peer_reviewed',
    },
}

ccle_issues = []
for fname, meta in EXPECTED_CCLE.items():
    fpath = CCLE_DIR / fname
    if not fpath.exists():
        ccle_issues.append(f'MISSING: {fname}')
        continue
    checksum = sha256_file(fpath)
    mtime    = file_mtime_iso(fpath)
    # Sanity-check the body of the file: row count + columns we depend on
    try:
        sample = pd.read_excel(fpath, sheet_name='CCLE_translocations_SvABA_20181')
        n_rows = len(sample)
        n_cell_lines = sample['CCLE_name'].nunique()
        cols = list(sample.columns)
    except Exception as e:
        ccle_issues.append(f'READ FAILED: {fname} — {e}')
        continue
    print(f'  OK  {fname}')
    print(f'      sha256 {checksum[:12]}…  mtime {mtime}  rows={n_rows:,}  cell_lines={n_cell_lines}')
    print(f'      columns: {cols}')
    provenance_records.append({
        'file':              str(fpath.relative_to(ROOT)),
        'source':            meta['source'],
        'reference_genome':  meta['reference_genome'],
        'caller':            meta['caller'],
        'source_confidence': meta['source_confidence'],
        'note':              meta['note'],
        'mtime':             mtime,
        'sha256':            checksum,
        'n_rows':            int(n_rows),
        'n_cell_lines':      int(n_cell_lines),
        'recorded_at':       RUN_TIMESTAMP,
    })

if ccle_issues:
    print(f'\nCCLE issues: {ccle_issues}')
else:
    print('\nAll CCLE files verified.')

  OK  CCLE_translocations_SvABA_20181221.xlsx
      sha256 0fca02bf2e9c…  mtime 2026-05-01T15:53:31Z  rows=75,344  cell_lines=328
      columns: ['CCLE_name', 'map_id', 'bp1', 'bp2', 'class', 'gene1', 'gene2', 'site1', 'site2', 'fusion', 'multi_sv_fusion', 'cosmic_fus']

All CCLE files verified.


## Phase 2 — Download TARGET-OS from UCSC Xena

Open-access; no authentication required.

In [7]:
import requests
from tqdm.notebook import tqdm

TARGET_DIR = ROOT / 'data' / 'raw' / 'target_os'
TARGET_DIR.mkdir(parents=True, exist_ok=True)

# NOTE: As of Jan 2024 UCSC Xena retired the HTSeq pipeline and consolidated
# survival + phenotype into a single clinical.tsv.gz. The legacy filenames
# (htseq_fpkm-uq.tsv.gz, survival.tsv, GDC_phenotype.tsv.gz) now 403 on the
# gdc-hub S3 bucket. We use the current STAR-aligned files instead.
XENA_FILES = {
    'TARGET-OS.star_fpkm-uq.tsv.gz': (
        'https://gdc.xenahubs.net/download/'
        'TARGET-OS.star_fpkm-uq.tsv.gz'
    ),
    'TARGET-OS.star_tpm.tsv.gz': (
        'https://gdc.xenahubs.net/download/'
        'TARGET-OS.star_tpm.tsv.gz'
    ),
    'TARGET-OS.clinical.tsv.gz': (
        'https://gdc.xenahubs.net/download/'
        'TARGET-OS.clinical.tsv.gz'
    ),
}


def _download(url: str, dest: Path, chunk_size: int = 1 << 20) -> str:
    if dest.exists():
        print(f'  EXISTS (skip): {dest.name}')
        return sha256_file(dest)
    print(f'  Downloading {dest.name} ...')
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size):
            f.write(chunk)
    ck = sha256_file(dest)
    print(f'  Done  sha256={ck[:12]}…')
    return ck


for fname, url in XENA_FILES.items():
    dest = TARGET_DIR / fname
    try:
        ck = _download(url, dest)
        provenance_records.append({
            'file': str(dest.relative_to(ROOT)),
            'source': url,
            'source_name': 'UCSC Xena TARGET hub',
            'downloaded_at': RUN_TIMESTAMP,
            'sha256': ck,
        })
    except Exception as exc:
        print(f'  FAILED {fname}: {exc}')
        print('  → Document this deviation in docs/data_provenance.md')

print('\nTARGET-OS download complete (or files already exist).')

  EXISTS (skip): TARGET-OS.star_fpkm-uq.tsv.gz
  EXISTS (skip): TARGET-OS.star_tpm.tsv.gz
  EXISTS (skip): TARGET-OS.clinical.tsv.gz

TARGET-OS download complete (or files already exist).


## Phase 3 — Download reference files (GENCODE v45, chrom sizes)

In [8]:
REF_DIR = ROOT / 'data' / 'raw' / 'reference'
REF_DIR.mkdir(parents=True, exist_ok=True)

REF_FILES = {
    'gencode.v45.annotation.gtf.gz': (
        'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/'
        'release_45/gencode.v45.annotation.gtf.gz'
    ),
    'hg38.chrom.sizes': (
        'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes'
    ),
}

for fname, url in REF_FILES.items():
    dest = REF_DIR / fname
    try:
        ck = _download(url, dest)
        provenance_records.append({
            'file': str(dest.relative_to(ROOT)),
            'source': url,
            'downloaded_at': RUN_TIMESTAMP,
            'sha256': ck,
        })
    except Exception as exc:
        print(f'  FAILED {fname}: {exc}')
        print('  → Document this deviation in docs/data_provenance.md')

print('Reference files ready.')

  EXISTS (skip): gencode.v45.annotation.gtf.gz
  EXISTS (skip): hg38.chrom.sizes
Reference files ready.


## Phase 4 — Write provenance sidecar (JSON-lines)

In [9]:
from src.utils import record_provenance

PROV_PATH = ROOT / 'data' / 'processed' / 'provenance.jsonl'
record_provenance(provenance_records, PROV_PATH)
print(f'Wrote {len(provenance_records)} provenance records to {PROV_PATH}')

Wrote 29 provenance records to /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/processed/provenance.jsonl


## Notebook summary

In [10]:
report_lines = [
    f'# Notebook 00 — Download Data Report',
    f'',
    f'Run: {RUN_TIMESTAMP}',
    f'',
    f'## Files verified',
    f'- DepMap files checked: {len(EXPECTED_DEPMAP)}',
    f'- DepMap missing: {missing}',
    f'- DepMap schema errors: {schema_errors}',
    f'- Literature files checked: {len(EXPECTED_LIT)}',
    f'- Literature issues: {lit_issues}',
    f'',
    f'## Downloads attempted',
    f'- TARGET-OS Xena files: {list(XENA_FILES.keys())}',
    f'- Reference files: {list(REF_FILES.keys())}',
    f'',
    f'## Provenance records written',
    f'- {len(provenance_records)} records → {PROV_PATH}',
    f'',
    f'## Status',
    f'- Missing DepMap: {"YES — do not proceed" if missing else "None"}',
    f'- Schema errors: {"YES — document in data_provenance.md" if schema_errors else "None"}',
]

report_path = ROOT / 'data' / 'processed' / 'notebook_00_report.md'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text('\n'.join(report_lines))
print('\n'.join(report_lines))

# Notebook 00 — Download Data Report

Run: 2026-05-01T16:59:14Z

## Files verified
- DepMap files checked: 8
- DepMap missing: []
- DepMap schema errors: []
- Literature files checked: 15
- Literature issues: []

## Downloads attempted
- TARGET-OS Xena files: ['TARGET-OS.star_fpkm-uq.tsv.gz', 'TARGET-OS.star_tpm.tsv.gz', 'TARGET-OS.clinical.tsv.gz']
- Reference files: ['gencode.v45.annotation.gtf.gz', 'hg38.chrom.sizes']

## Provenance records written
- 29 records → /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/processed/provenance.jsonl

## Status
- Missing DepMap: None
- Schema errors: None
